<a href="https://colab.research.google.com/github/joseguilhermemarinho/big_data_proj/blob/main/codigo/ingestao_av1_big_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import requests
import pandas as pd
from datetime import datetime
import pytz
import os

url = "https://dados.mobilidade.rio/gps/sppo"
fuso_horario = pytz.timezone('America/Recife')

In [34]:
response = requests.get(url)

if response.status_code == 200:
  dados_brutos = response.json()
  df_gps = pd.DataFrame(dados_brutos)

  df_gps['momento_extracao'] = datetime.now(fuso_horario).strftime('%Y-%m-%d %H:%M:%S')

  nome_arquivo = "gps_sppo_bruto.csv"
  df_gps.to_csv(nome_arquivo, index=False)

  print(f"Foram ingeridos {len(df_gps)} registros de GPS.")
  display(df_gps.head())

else:
  print(f"Erro ao acessar a API. Status: {response.status_code}")

Foram ingeridos 489590 registros de GPS.


,ordem,latitude,longitude,datahora,velocidade,linha,datahoraenvio,datahoraservidor,momento_extracao
0,B71083,"-22,90589","-43,19067",1775587402000,0,247,1775587413000,1775587419000,2026-04-07 16:43:46
1,B71018,"-22,91505","-43,22887",1775587400000,24,247,1775587413000,1775587419000,2026-04-07 16:43:46
2,B71051,"-22,8834","-43,29652",1775587401000,0,457,1775587413000,1775587419000,2026-04-07 16:43:46
3,B71005,"-22,9442","-43,18185",1775587403000,0,457,1775587413000,1775587419000,2026-04-07 16:43:46
4,B71031,"-22,90006","-43,28035",1775587403000,36,457,1775587413000,1775587419000,2026-04-07 16:43:46


aacessa os dados via API, acrescenta o momento do salvamento e cria o primeiro arquivo bruto. porém, o arquivo bruto é muito grande, precisamos garantir que não vai dar trabalho à equipe de transformação.

In [35]:
tamanho_mb = os.path.getsize("gps_sppo_bruto.csv") / (1024 * 1024)
print(f"O arquivo original tem: {tamanho_mb:.2f} MB")

nome_amostra_github = "amostra_gps_sppo_bruto.csv"
df_gps.head(2000).to_csv(nome_amostra_github, index=False)

print(f"Amostra reduzida com sucesso")

O arquivo original tem: 46.57 MB
Amostra reduzida com sucesso


a extração completa resultou em quase meio milhão de registros, o que gera um arquivo muito pesado. a redução para 2000 linhas foi aplicada para cumprir estritamente as diretrizes do projeto, que determinam que a pasta /dados deve conter apenas amostras pequenas e que arquivos grandes não devem ser commitados. isso garante a performance do repositório no GitHub e evita bloqueios de tamanho.

In [36]:
df = df_gps.copy()

colunas_timestamp = ['datahora', 'datahoraenvio', 'datahoraservidor']

for col in colunas_timestamp:
  df[col] = pd.to_datetime(pd.to_numeric(df[col], errors='coerce'), unit='ms', utc=True)
  df[col] = df[col].dt.tz_convert('America/Sao_Paulo')

print("Exemplo após conversão:")
print(df[['datahora', 'datahoraenvio', 'datahoraservidor']].head())

Exemplo após conversão:
                   datahora             datahoraenvio  \
0 2026-04-07 15:43:22-03:00 2026-04-07 15:43:33-03:00   
1 2026-04-07 15:43:20-03:00 2026-04-07 15:43:33-03:00   
2 2026-04-07 15:43:21-03:00 2026-04-07 15:43:33-03:00   
3 2026-04-07 15:43:23-03:00 2026-04-07 15:43:33-03:00   
4 2026-04-07 15:43:23-03:00 2026-04-07 15:43:33-03:00   

           datahoraservidor  
0 2026-04-07 15:43:39-03:00  
1 2026-04-07 15:43:39-03:00  
2 2026-04-07 15:43:39-03:00  
3 2026-04-07 15:43:39-03:00  
4 2026-04-07 15:43:39-03:00  


Os dados nos são enviados em tempo real. A cada vez que executamos a célula que puxa os dados da api, são enviados novos dados para nós.

As colunas de datahora, datahoraenvio e datahoraservidor estão todas em um timestamp unix normalmente utilizados em sistema iot, o que acontece com os dados desse datalake.

Por isso, a conversão desse timestamp para o formato de yyyy-mm-dd hh:mm:ss - GMT-3 é muito importante para que nós possamos visualizar melhor e também entendermos perfeitamente a "tradução" do timestamp.

In [37]:
#Criando coluinas derivadas a partir do datahora

df['data'] = df['datahora'].dt.date
df['hora'] = df['datahora'].dt.hour
df['minuto'] = df['datahora'].dt.minute
df['dia_semana'] = df['datahora'].dt.day_name()

#Calculando latencia de envio: tempo entre captura do GPS e envio (ms)
df['latencia_envio_s'] = (
    df['datahoraenvio'] - df['datahora']
).dt.total_seconds()

#Calculando agora latencia do servidor: tempo entre envio e chegada ao servidor (ms)
df['latencia_servidor_s'] = (
    df['datahoraservidor'] - df['datahoraenvio']
).dt.total_seconds()

print("\nLatências calculadas:")
print(df[['latencia_envio_s', 'latencia_servidor_s']].describe())


Latências calculadas:
       latencia_envio_s  latencia_servidor_s
count     489590.000000        489590.000000
mean          16.592898            16.357742
std          106.314112             9.164598
min            0.000000          -703.000000
25%            5.000000             9.000000
50%            8.000000            16.000000
75%           10.000000            24.000000
max         4698.000000            46.000000


In [38]:
print("\nNulos antes da limpeza:")
print(df.isnull().sum())


Nulos antes da limpeza:
ordem                  0
latitude               0
longitude              0
datahora               0
velocidade             0
linha                  0
datahoraenvio          0
datahoraservidor       0
momento_extracao       0
data                   0
hora                   0
minuto                 0
dia_semana             0
latencia_envio_s       0
latencia_servidor_s    0
dtype: int64


Por mais que não existam números nulos antes da limpeza, é importante lembrar que os dados que estamos tratando são todos em tempo real. A cada nova execução de linha de código, novos dados chegam.

Por isso, precisamos normalizar, padronizar e estruturar corretamente cada dado novo que chega pois isso facilitará e muito na nossa análise da camada gold.

In [39]:
# Substituir vírgula por ponto ANTES de converter para numérico
for col in ['latitude', 'longitude', 'velocidade']:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop de linhas com nulos em colunas críticas
colunas_criticas = ['ordem', 'latitude', 'longitude', 'datahora', 'velocidade']
df = df.dropna(subset=colunas_criticas)

# Remover coordenadas fora do Brasil
df = df[
    (df['latitude'].between(-34, 5)) &
    (df['longitude'].between(-74, -28))
]

# Remover velocidades negativas
df = df[df['velocidade'] >= 0]

print("Shape após limpeza:", df.shape)
print("\nExemplo de coordenadas após correção:")
print(df[['ordem', 'latitude', 'longitude', 'velocidade']].head())

Shape após limpeza: (489590, 15)

Exemplo de coordenadas após correção:
    ordem  latitude  longitude  velocidade
0  B71083 -22.90589  -43.19067           0
1  B71018 -22.91505  -43.22887          24
2  B71051 -22.88340  -43.29652           0
3  B71005 -22.94420  -43.18185           0
4  B71031 -22.90006  -43.28035          36


In [40]:
#Salvando na camada silver

os.makedirs('/content/silver', exist_ok=True)

df.to_parquet('/content/silver/sppo_silver.parquet', index=False)
df.to_csv('/content/silver/sppo_silver.csv', index=False)

df_amostra = df.sample(n=1000, random_state=42)
df_amostra.to_csv('/content/silver/sppo_amostra.csv', index=False)

print("Salvo em /content/silver/")
print("Shape final:", df.shape)

Salvo em /content/silver/
Shape final: (489590, 15)


In [41]:
df.head()

,ordem,latitude,longitude,datahora,velocidade,linha,datahoraenvio,datahoraservidor,momento_extracao,data,hora,minuto,dia_semana,latencia_envio_s,latencia_servidor_s
0,B71083,-22.90589,-43.19067,2026-04-07 15:43:22-03:00,0,247,2026-04-07 15:43:33-03:00,2026-04-07 15:43:39-03:00,2026-04-07 16:43:46,2026-04-07,15,43,Tuesday,11.0,6.0
1,B71018,-22.91505,-43.22887,2026-04-07 15:43:20-03:00,24,247,2026-04-07 15:43:33-03:00,2026-04-07 15:43:39-03:00,2026-04-07 16:43:46,2026-04-07,15,43,Tuesday,13.0,6.0
2,B71051,-22.88340,-43.29652,2026-04-07 15:43:21-03:00,0,457,2026-04-07 15:43:33-03:00,2026-04-07 15:43:39-03:00,2026-04-07 16:43:46,2026-04-07,15,43,Tuesday,12.0,6.0
3,B71005,-22.94420,-43.18185,2026-04-07 15:43:23-03:00,0,457,2026-04-07 15:43:33-03:00,2026-04-07 15:43:39-03:00,2026-04-07 16:43:46,2026-04-07,15,43,Tuesday,10.0,6.0
4,B71031,-22.90006,-43.28035,2026-04-07 15:43:23-03:00,36,457,2026-04-07 15:43:33-03:00,2026-04-07 15:43:39-03:00,2026-04-07 16:43:46,2026-04-07,15,43,Tuesday,10.0,6.0


Com base no fato de que temos dados que não sempre atualizados em tempo real, termos organizado a questão do timestamp, termos latitude e longitude e conseguirmos criar novas colunas com dados que podem nos ajudar em análises mais complexas, percebemos alguns insights:



1.   Podemos monitorar pontos de trânsito dentro da cidade, inclusive conseguindo apontar o ponto específico por conta da latitude e longitude.
2.   Podemos também monitorar linhas de transporte público, identificando mudanças que poderiam acontecer, como remoção ou adição de novas linhas.
3. Podemos criar gráficos para identificar horários de pico e velocidade média de cada local, gerando insights de melhoria de infraestrutura da cidade e mobilidade urbana.

